## 1. Setup and Configuration

In [37]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from datetime import datetime

In [38]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../../.env')

# Validate credentials
required_vars = ['DATABRICKS_HOST', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

✓ Environment configured


## 2. Analysis Parameters

### Product Groups
1. **Ariel Mirai Regular (本体通常)**: TFI 7/1-8/31, Repeat 7/1-11/30
2. **Ariel Mirai Large (本体大)**: TFI 7/1-8/31, Repeat 7/1-11/30  
3. **Ariel Gel (本体)**: TFI 9/1-9/30, Repeat 9/1-12/31

### Channel Exclusions
- FAMILYMART, LAWSON, SEVEN ELEVEN

### Customers (All IDPOS retailers)
- cds_8005 (TSURUHA)
- cds_8006 (TOMODS)
- cds_8007 (SAPPORO DRUG)
- cds_8008 (KOHNAN)
- cds_8009 (FUJI YAKUHIN)
- cds_8010 (TRIAL)
- cds_8011 (CHUBU YAKUHIN)
- cds_8012 (CAINZ)
- cds_8013 (SUGI YAKKYOKU)

In [39]:
# Define analysis parameters
analysis_config = [
    {
        'product_name': 'Ariel Mirai Regular',
        'sub_brand': 'ｱﾘｴｰﾙﾐﾗｲ',
        'size_filter': '本体通常',
        'tfi_start': '2025-07-01',
        'tfi_end': '2025-08-31',
        'repeat_start': '2025-07-01',
        'repeat_end': '2025-11-30'
    },
    {
        'product_name': 'Ariel Mirai Large',
        'sub_brand': 'ｱﾘｴｰﾙﾐﾗｲ',
        'size_filter': '本体大',
        'tfi_start': '2025-07-01',
        'tfi_end': '2025-08-31',
        'repeat_start': '2025-07-01',
        'repeat_end': '2025-11-30'
    },
    {
        'product_name': 'Ariel Gel',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙ',
        'size_filter': '本体通常', 
        'tfi_start': '2025-09-01',
        'tfi_end': '2025-09-30',
        'repeat_start': '2025-09-01',
        'repeat_end': '2025-12-31'
    }
]

# Customer filters (all IDPOS retailers, excluding CVS)
customer_codes = ['cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009', 
                  'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013']
customer_filter_sql = "', '".join(customer_codes)

# Category
category = 'Laundry'

print(f"✓ Parameters configured")
print(f"  Product groups: {len(analysis_config)}")
print(f"  Customers: {len(customer_codes)}")
print(f"  Category: {category}")

✓ Parameters configured
  Product groups: 3
  Customers: 9
  Category: Laundry


## 3. Define Repeat Rate Calculation Function

In [40]:
def calculate_repeat_rate(connection, config):
    """
    Calculate repeat rate for a specific product configuration.
    
    Logic:
    1. Identify TFI shoppers: Shoppers who purchased the target product in TFI period (記録trial_date)
    2. Identify Next Purchase shoppers: TFI shoppers whose 2nd category purchase is the target product
    3. Identify Any Repeat shoppers: TFI shoppers who purchased target product at any time after their trial_date
    4. Calculate both repeat rates
    
    Note: Repeat period starts from tfi_start but excludes the first purchase (trial_date) using diff > 0 logic
    
    Returns:
    - tfi_shoppers: Count of unique shoppers in TFI period
    - next_purchase_shoppers: Count where target product is 2nd purchase
    - any_repeat_shoppers: Count who bought target product anytime after TFI
    - next_purchase_rate: Percentage (2nd purchase only)
    - any_repeat_rate: Percentage (any purchase)
    """
    
    # Build SQL query
    query = f"""
    WITH tfi_purchasers AS (
        -- Step 1: Identify shoppers who purchased target product during TFI period
        SELECT DISTINCT
            idpos.shopper_key AS shopper_id,
            MIN(sales_period_group_end_date_part) AS trial_date
        FROM
            cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
            LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
            LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
        WHERE
            jp_category_name = '{category}'
            AND jp_sub_brand_alter_lang_name = '{config['sub_brand']}'
            AND jp_segment_4_name LIKE '{config['size_filter']}'
            AND idpos.data_provider_code_part IN ('{customer_filter_sql}')
            AND sales_period_group_end_date_part BETWEEN '{config['tfi_start']}' AND '{config['tfi_end']}'
            AND shopper.member_ind = 'Y'
        GROUP BY idpos.shopper_key
    ),
    all_category_purchases_after_tfi AS (
        -- Get ALL category purchases after each shopper's trial_date (diff > 0)
        SELECT
            idpos.shopper_key AS shopper_id,
            sales_period_group_end_date_part AS purchase_date,
            jp_sub_brand_alter_lang_name,
            DATEDIFF(sales_period_group_end_date_part, tfi.trial_date) AS diff,
            ROW_NUMBER() OVER (PARTITION BY idpos.shopper_key ORDER BY sales_period_group_end_date_part) AS purchase_number
        FROM
            cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
            LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
            LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
            INNER JOIN tfi_purchasers tfi ON idpos.shopper_key = tfi.shopper_id
        WHERE
            jp_category_name = '{category}'
            AND idpos.data_provider_code_part IN ('{customer_filter_sql}')
            AND sales_period_group_end_date_part > tfi.trial_date
            AND sales_period_group_end_date_part <= '{config['repeat_end']}'
            AND shopper.member_ind = 'Y'
    ),
    next_purchase_shoppers AS (
        -- Shoppers whose 2nd purchase (in category) is the target product
        SELECT DISTINCT shopper_id
        FROM all_category_purchases_after_tfi
        WHERE purchase_number = 2
            AND jp_sub_brand_alter_lang_name = '{config['sub_brand']}'
    ),
    any_repeat_shoppers AS (
        -- Shoppers who bought target product at ANY time after TFI (diff > 0)
        SELECT DISTINCT shopper_id
        FROM all_category_purchases_after_tfi
        WHERE jp_sub_brand_alter_lang_name = '{config['sub_brand']}'
    )
    SELECT
        (SELECT COUNT(DISTINCT shopper_id) FROM tfi_purchasers) AS tfi_shoppers,
        (SELECT COUNT(DISTINCT shopper_id) FROM next_purchase_shoppers) AS next_purchase_shoppers,
        (SELECT COUNT(DISTINCT shopper_id) FROM any_repeat_shoppers) AS any_repeat_shoppers
    """
    
    # Execute query
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchone()
        
        tfi_shoppers = result[0] if result[0] else 0
        next_purchase_shoppers = result[1] if result[1] else 0
        any_repeat_shoppers = result[2] if result[2] else 0
        
        next_purchase_rate = (next_purchase_shoppers / tfi_shoppers * 100) if tfi_shoppers > 0 else 0
        any_repeat_rate = (any_repeat_shoppers / tfi_shoppers * 100) if tfi_shoppers > 0 else 0
        
        return {
            'product_name': config['product_name'],
            'tfi_period': f"{config['tfi_start']} to {config['tfi_end']}",
            'repeat_period': f"{config['repeat_start']} to {config['repeat_end']}",
            'tfi_shoppers': tfi_shoppers,
            'next_purchase_shoppers': next_purchase_shoppers,
            'next_purchase_rate': next_purchase_rate,
            'any_repeat_shoppers': any_repeat_shoppers,
            'any_repeat_rate': any_repeat_rate
        }

print("✓ Function defined")

✓ Function defined


## 4. Execute Analysis for All Product Groups

In [41]:
# Connect to Databricks and run analysis
results = []

with sql.connect(
    server_hostname=os.getenv("DATABRICKS_HOST"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    
    for config in analysis_config:
        print(f"\nProcessing: {config['product_name']}...")
        result = calculate_repeat_rate(connection, config)
        results.append(result)
        print(f"  ✓ TFI Shoppers: {result['tfi_shoppers']:,}")
        print(f"  ✓ Next Purchase (2nd only): {result['next_purchase_shoppers']:,} ({result['next_purchase_rate']:.2f}%)")
        print(f"  ✓ Any Repeat (any time): {result['any_repeat_shoppers']:,} ({result['any_repeat_rate']:.2f}%)")
        print(f"  ✓ Difference: {result['any_repeat_shoppers'] - result['next_purchase_shoppers']:,} shoppers bought competitors first, then returned")

print("\n" + "="*60)
print("✓ All analyses completed successfully")
print("="*60)


Processing: Ariel Mirai Regular...
  ✓ TFI Shoppers: 31,278
  ✓ Next Purchase (2nd only): 4,783 (15.29%)
  ✓ Any Repeat (any time): 10,843 (34.67%)
  ✓ Difference: 6,060 shoppers bought competitors first, then returned

Processing: Ariel Mirai Large...
  ✓ TFI Shoppers: 87,132
  ✓ Next Purchase (2nd only): 16,326 (18.74%)
  ✓ Any Repeat (any time): 36,273 (41.63%)
  ✓ Difference: 19,947 shoppers bought competitors first, then returned

Processing: Ariel Gel...
  ✓ TFI Shoppers: 191,751
  ✓ Next Purchase (2nd only): 44,146 (23.02%)
  ✓ Any Repeat (any time): 87,074 (45.41%)
  ✓ Difference: 42,928 shoppers bought competitors first, then returned

✓ All analyses completed successfully


## 5. Summary Results Table

In [42]:
# Create results dataframe
results_df = pd.DataFrame(results)

# Display formatted table
print("\n" + "="*90)
print("ARIEL BOTTLE PRODUCTS - REPEAT RATE COMPARISON")
print("="*90)
print(f"\nAnalysis Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"Channels: All channels excluding FAMILYMART, LAWSON, SEVEN ELEVEN")
print(f"Customers: All 9 IDPOS retailers aggregated")
print("\nTwo Metrics Compared:")
print("  1. Next Purchase Rate: Target product is shopper's 2nd category purchase (stricter)")
print("  2. Any Repeat Rate: Target product purchased anytime after TFI (broader)")
print("\n")

# Format for display
display_df = results_df.copy()
display_df['tfi_shoppers'] = display_df['tfi_shoppers'].apply(lambda x: f"{x:,}")
display_df['next_purchase_shoppers'] = display_df['next_purchase_shoppers'].apply(lambda x: f"{x:,}")
display_df['next_purchase_rate'] = display_df['next_purchase_rate'].apply(lambda x: f"{x:.2f}%")
display_df['any_repeat_shoppers'] = display_df['any_repeat_shoppers'].apply(lambda x: f"{x:,}")
display_df['any_repeat_rate'] = display_df['any_repeat_rate'].apply(lambda x: f"{x:.2f}%")

# Reorder columns for clarity
display_df = display_df[['product_name', 'tfi_shoppers', 
                         'next_purchase_shoppers', 'next_purchase_rate',
                         'any_repeat_shoppers', 'any_repeat_rate']]

display_df


ARIEL BOTTLE PRODUCTS - REPEAT RATE COMPARISON

Analysis Date: 2026-01-07
Channels: All channels excluding FAMILYMART, LAWSON, SEVEN ELEVEN
Customers: All 9 IDPOS retailers aggregated

Two Metrics Compared:
  1. Next Purchase Rate: Target product is shopper's 2nd category purchase (stricter)
  2. Any Repeat Rate: Target product purchased anytime after TFI (broader)




,product_name,tfi_shoppers,next_purchase_shoppers,next_purchase_rate,any_repeat_shoppers,any_repeat_rate
0,Ariel Mirai Regular,"31,278","4,783",15.29%,"10,843",34.67%
1,Ariel Mirai Large,"87,132","16,326",18.74%,"36,273",41.63%
2,Ariel Gel,"191,751","44,146",23.02%,"87,074",45.41%


## 6. Export Results

In [43]:
# Export to Excel
output_file = 'ariel_bottle_repeat_rate_results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Main results
    results_df.to_excel(writer, sheet_name='Repeat_Rate_Results', index=False)
    
    # Analysis parameters
    params_df = pd.DataFrame(analysis_config)
    params_df.to_excel(writer, sheet_name='Parameters', index=False)
    
    # Metadata
    metadata_df = pd.DataFrame({
        'Item': ['Analysis Date', 'Category', 'Channels Excluded', 'Number of Retailers', 'Retailers'],
        'Value': [
            datetime.now().strftime('%Y-%m-%d'),
            category,
            'FAMILYMART, LAWSON, SEVEN ELEVEN',
            str(len(customer_codes)),
            ', '.join(customer_codes)
        ]
    })
    metadata_df.to_excel(writer, sheet_name='Metadata', index=False)

print(f"\n✓ Results exported to: {output_file}")
print("\nReady for MCC comparison")


✓ Results exported to: ariel_bottle_repeat_rate_results.xlsx

Ready for MCC comparison


## 7. Key Findings Summary

**For Sales Team:**

This analysis compares two different repeat rate definitions:

### Metric 1: Next Purchase Rate (Stricter Definition)
- **Next Purchase Shoppers**: TFI shoppers whose **2nd category purchase** is the target product
- Matches your original template logic with `purchase_number = 2`
- Excludes shoppers who bought competitors first
- More conservative metric showing immediate brand loyalty

### Metric 2: Any Repeat Rate (Broader Definition)  
- **Any Repeat Shoppers**: TFI shoppers who bought the target product **at any time** after TFI
- Includes shoppers who may have tried competitors but returned to the target product
- More inclusive metric showing overall brand retention

### The Difference
The gap between these two metrics represents shoppers who:
1. Bought the target product during TFI (e.g., Ariel Gel 本体通常)
2. Bought a **competitor** as their 2nd category purchase
3. Later **returned** to the target product (3rd, 4th, or later purchase)

**Calculation Method:**
- Aggregated across all 9 IDPOS retailers (not per-retailer breakdown)
- Excludes convenience stores (FAMILYMART, LAWSON, SEVEN ELEVEN)
- Uses same shopper for both TFI and repeat periods (cohort-based analysis)
- Both metrics use DISTINCT shopper counts

**Comparison with MCC:**
Clarify with the sales team which definition MCC uses for accurate comparison.